# 04 Outbreak Investigation Workflow: From Line List to SitRep

Legionnaires' disease has broken out at Pine and Cypress Nursing Home, and your supervisor wants the first SitRep within two hours.

This lesson walks through the whole thing: **read in → summary metrics → the person/time/place trio → case classification → structured output**.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .
    !pip install -q python-docx python-pptx fpdf2

In [ ]:
# --- Step 1: Read in and prepare the data ---
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (avoid CJK labels rendering as boxes) --
# Scan the system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# Convert dates
date_cols = [
    "facility_admission_date", "symptom_onset_date",
    "hospitalization_date", "death_date", "notification_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Derive columns
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["age_group"] = pd.cut(
    df["age"], bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)
comorbidity_cols = [
    "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "comorbidity_copd", "immunosuppressed",
]
df["n_comorbidities"] = df[comorbidity_cols].sum(axis=1)

print(f"Data shape: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# --- Step 2: Summary metrics ---
total = len(df)
infected = df["infected"].sum()
confirmed = (df["case_classification"] == "confirmed").sum()
probable = (df["case_classification"] == "probable").sum()
hospitalized = df["hospitalized"].sum()
icu = df["icu_admission"].sum()
deaths = (df["outcome"] == "dead").sum()

print("=" * 50)
print("Pine and Cypress Nursing Home Legionnaires' Disease Cluster — SitRep")
print("=" * 50)
print(f"Total residents: {total}")
print(f"Infected: {infected} (attack rate {infected/total:.1%})")
print(f"  Confirmed: {confirmed}   Probable: {probable}")
print(f"Hospitalized: {hospitalized} (hospitalization rate {hospitalized/infected:.1%})")
print(f"ICU: {icu} (ICU rate {icu/hospitalized:.1%})")
print(f"Deaths: {deaths} (CFR {deaths/infected:.1%})")

In [ ]:
# --- Step 3: Person ---
cases = df[df["infected"] == 1]

print("=== Demographic characteristics (infected) ===")
print(f"Median age: {cases['age'].median():.0f} years"
      f" (range {cases['age'].min()}-{cases['age'].max()})")
print(f"Proportion male: {(cases['sex'] == 'M').mean():.1%}")

print(f"\n--- Age group distribution ---")
age_dist = cases["age_group"].value_counts().sort_index()
for grp, n in age_dist.items():
    print(f"  {grp}: {n} ({n/len(cases):.1%})")

print(f"\n--- Comorbidity distribution ---")
for col in comorbidity_cols:
    label = col.replace("comorbidity_", "").upper()
    n = int(cases[col].sum())
    print(f"  {label}: {n} ({n/len(cases):.1%})")

In [ ]:
# --- Step 4: Time — epidemic curve ---
import matplotlib.dates as mdates

# groupby("symptom_onset_date"): group by onset date
# .size(): count the rows in each group (= cases that day), like GROUP BY + COUNT(*)
# .rename("cases"): name the result column "cases" for easier reference later
daily = cases.groupby("symptom_onset_date").size().rename("cases")

# -- Fill in the full date range (including 3 days before the outbreak as a "baseline period") --
# The raw data only has dates that had cases; a day with 0 cases won't appear in the groupby result
# reindex can "fill in" the missing dates, using fill_value=0
date_range = pd.date_range(
    daily.index.min() - pd.Timedelta(days=3),  # extend 3 days earlier (show the pre-outbreak baseline)
    daily.index.max() + pd.Timedelta(days=1),  # extend 1 day later (so the last day isn't clipped)
    freq="D",                                   # freq="D" means one point per day
)
daily = daily.reindex(date_range, fill_value=0)  # fill days with no cases as 0

# -- Build the chart --
# plt.subplots() returns two objects at once:
# fig = the whole canvas (controls overall size, resolution, saving)
# ax  = the plotting area (controls axes, title, bars, lines, etc.)
fig, ax = plt.subplots(figsize=(10, 4))  # 10 inches wide, 4 inches tall

ax.bar(daily.index, daily.values, width=1.0,
       color="#2c7fb8", edgecolor="white", linewidth=0.5)
# width=1.0 makes the bars touch (standard for an epidemic curve, no gaps)

ax.set_title("Pine and Cypress Nursing Home Legionnaires' Disease Epidemic Curve, by Onset Date, January 2026",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")

# DateFormatter("%m/%d"): set the x-axis date display format
# %m = month (01–12), %d = day (01–31), giving e.g. "01/12"
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
# DayLocator(interval=2): place a tick every 2 days (avoid overlapping labels)
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
# auto-rotate the date labels 45 degrees to prevent overlap
fig.autofmt_xdate(rotation=45)

# Leave a half-day (12-hour) margin on the far left and right so the first and last bars aren't clipped
ax.set_xlim(daily.index.min() - pd.Timedelta(hours=12),
            daily.index.max() + pd.Timedelta(hours=12))
ax.set_ylim(bottom=0)  # y-axis starts at 0
# MaxNLocator(integer=True): show only integer ticks on the y-axis (you can't have 0.5 cases)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
# Remove the top and right spines (cleaner look, standard for epidemiology papers)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()  # auto-adjust spacing so the title and labels aren't cut off
plt.show()

print(f"Outbreak period: {cases['symptom_onset_date'].min().date()} – {cases['symptom_onset_date'].max().date()}")
# idxmax(): find the "index" (date) of the maximum value, not the max value itself
# daily.max() is the maximum value (the case count)
print(f"Peak day: {daily.idxmax().date()} ({daily.max()} cases)")


In [ ]:
# --- Step 5: Place — attack rate by wing ---
# -- Group by floor + wing, computing all metrics at once --
# .agg() lets you apply different aggregation functions to different columns
# Format: new_column_name=("source_column", "aggregation function or lambda")
wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(
        # "size" counts the total rows in each group (= total residents of that wing, infected and not)
        residents=("case_id", "size"),
        # "infected" is a 0/1 column, so "sum" gives the number infected
        infected=("infected", "sum"),
        # lambda for custom logic: count the values equal to "dead" in the outcome column
        deaths=("outcome", lambda x: (x == "dead").sum()),
    )
    .reset_index()
    # reset_index() turns the groupby keys (floor, wing) from the "index"
    # back into ordinary columns, so we can reference them by name below
)

# Compute attack rate and CFR, multiply by 100 to make percentages, round to 1 decimal
wing_stats["AR%"] = (wing_stats["infected"] / wing_stats["residents"] * 100).round(1)
wing_stats["CFR%"] = (wing_stats["deaths"] / wing_stats["infected"] * 100).round(1)
# Join the floor (number) and wing (letter) into one label, e.g. 1 + "A" → "1A"
# astype(str) first converts the integer to a string so it can be concatenated with the letter
wing_stats["label"] = wing_stats["floor"].astype(str) + wing_stats["wing"]

print("=== Outbreak summary by wing ===")
# to_string(index=False) prints without the left-side index numbers
print(wing_stats[["label", "residents", "infected", "AR%", "deaths", "CFR%"]]
      .to_string(index=False))


In [ ]:
# --- Step 6: Stratified summary by case classification ---
# Group by case classification (confirmed / probable / not_a_case) and compute per-tier metrics
# We don't add .reset_index() here, so case_classification stays as the index,
# making the printout more intuitive (the classification name appears in the leftmost column)
classification = (
    df.groupby("case_classification")
    .agg(
        n=("case_id", "size"),                              # count per tier
        hospitalized=("hospitalized", "sum"),               # hospitalized count (sum of 0/1 column)
        icu=("icu_admission", "sum"),                       # ICU count
        deaths=("outcome", lambda x: (x == "dead").sum()),  # death count
    )
)

# Compute hospitalization rate: hospitalized / tier count × 100
classification["hosp_rate%"] = (
    classification["hospitalized"] / classification["n"] * 100
).round(1)

print("=== Stratified by case classification ===")
# .to_string() with no arguments keeps the index (= case_classification names) for easy reference
print(classification.to_string())


In [ ]:
# --- Step 7: Wrap into a rerunnable function ---
def generate_sitrep(csv_path):
    """Produce a SitRep summary dictionary from a CSV.

    Each day, just rerun this function with the latest CSV to auto-update every metric.
    It returns a dict rather than printing directly, because a dict can be consumed directly by the later Step 8 (report output).
    """
    df = pd.read_csv(csv_path)
    for col in ["symptom_onset_date", "hospitalization_date",
                "death_date", "notification_date"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

    total = len(df)
    # int() converts numpy.int64 to a native Python int
    # Why: pandas / numpy .sum() returns a numpy integer type (numpy.int64)
    # Putting that straight into a dict and serializing to JSON raises a JSON serialization error
    infected = int(df["infected"].sum())
    deaths = int((df["outcome"] == "dead").sum())

    return {
        "total_residents": total,
        "infected": infected,
        # round(value, decimals): round to the given number of decimal places
        "attack_rate": round(infected / total * 100, 1),
        "deaths": deaths,
        # guard: if infected == 0 (no cases yet), return 0 to avoid ZeroDivisionError
        "cfr": round(deaths / infected * 100, 1) if infected else 0,
        "hospitalized": int(df["hospitalized"].sum()),
        "icu": int(df["icu_admission"].sum()),
    }

# sitrep is a Python dict, ready to hand to the report-output function in Step 8
sitrep = generate_sitrep("data/synthetic/legionella_outbreak.csv")
print(sitrep)


## Step 8: Produce a professional report

The dictionary returned by `generate_sitrep()` is your data layer, but your supervisor wants a polished report. Here are four professional output formats:

| Format | Best for | Python package |
|------|---------|------------|
| Interactive dashboard | Real-time viewing, internal team discussion | plotly (already installed) |
| Word (.docx) | Handing to a manager, email attachment | python-docx |
| Slides (.pptx) | Investigation meeting presentations | python-pptx |
| PDF | Formal filing, printing | fpdf2 |

In [ ]:
# --- Step 8 shared setup: save the chart and create the output folder ---
import pathlib
from io import BytesIO
from datetime import datetime

# exist_ok=True: don't raise if the folder already exists (you can rerun this line safely)
pathlib.Path("output").mkdir(exist_ok=True)

# -- BytesIO: save the figure into an "in-memory virtual file" --
# Normally fig.savefig("epicurve.png") writes to disk;
# BytesIO() opens a "fake file" in memory that behaves exactly like a real file object,
# but the data lives only in RAM — no disk space used, nothing to clean up afterward.
# Bonus: DOCX / PPTX add_picture() both accept BytesIO objects,
#       and the same figure can be reused (just remember to .seek(0) before each use).
epicurve_buf = BytesIO()
fig.savefig(epicurve_buf, format="png", dpi=150, bbox_inches="tight")
# seek(0): move the read cursor back to the very start of the buffer
# Analogy: rewind a tape to the beginning so you can play it from the top
# If you read without seek(0), you'd read from the end and get empty data
epicurve_buf.seek(0)

# strftime format string: %Y=4-digit year, %m=2-digit month, %d=2-digit day, %H=hour (24h), %M=minute
report_time = datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Report time: {report_time}")


In [ ]:
# --- 8a: Interactive dashboard (Plotly Dashboard) ---
# In JupyterLab / Colab the chart is interactive; the static Jupyter Book page auto-generates a screenshot

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# make_subplots builds a 2×2 grid of subplots
# specs sets each subplot's type:
#   "indicator" = a numeric indicator (large-font KPI display)
#   "xy"        = an ordinary x-y plot (bar chart, line chart, etc.)
dashboard = make_subplots(
    rows=2, cols=2,
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}],
        [{"type": "xy"}, {"type": "xy"}],
    ],
    subplot_titles=("", "", "Epidemic curve (by onset date)", "Attack rate by wing"),
    vertical_spacing=0.15,   # vertical gap between top and bottom subplots (0–1, a ratio)
    horizontal_spacing=0.1,
)

# -- Top-left: infected-count KPI indicator --
# go.Indicator is Plotly's "dashboard indicator" shape, made to show one big number + supporting info
dashboard.add_trace(
    go.Indicator(
        mode="number+delta",
        value=infected,
        title={"text": "Infected (attack rate)"},
        # number.suffix appends text after the number (the attack rate in parentheses)
        number={"suffix": f"  ({infected/total:.1%})"},
        delta={"reference": 0, "position": "bottom"},
    ),
    row=1, col=1,  # place in row 1, column 1 (top-left)
)

# -- Top-right: death-count KPI indicator --
dashboard.add_trace(
    go.Indicator(
        mode="number+delta",
        value=deaths,
        title={"text": "Deaths (CFR)"},
        number={"suffix": f"  ({deaths/infected:.1%})"},
        delta={"reference": 0, "position": "bottom"},
    ),
    row=1, col=2,
)

# -- Bottom-left: epidemic curve --
daily_cases = cases.groupby("symptom_onset_date").size()
dashboard.add_trace(
    go.Bar(
        x=daily_cases.index,    # x-axis: onset date
        y=daily_cases.values,   # y-axis: daily case count
        marker_color="#D97757",
        name="Daily cases",
    ),
    row=2, col=1,
)

# -- Bottom-right: attack rate by wing (horizontal bar chart) --
dashboard.add_trace(
    go.Bar(
        y=wing_stats["label"],  # y-axis holds wing names (the category axis of a horizontal bar chart)
        x=wing_stats["AR%"],    # x-axis holds attack-rate values
        orientation="h",        # "h" = horizontal
        marker_color="#6A9BCC",
        name="Attack rate %",
        # show a value label just outside each bar
        text=wing_stats["AR%"].apply(lambda v: f"{v:.1f}%"),
        textposition="outside",
    ),
    row=2, col=2,
)

dashboard.update_layout(
    title_text=f"Pine and Cypress Nursing Home Legionnaires' SitRep Dashboard ({report_time})",
    height=600,
    showlegend=False,      # hide the legend (the subplot titles are explanation enough)
    template="plotly_white",
)
dashboard.show()


In [ ]:
# --- 8b: Word document (DOCX) ---
# Note: the package is called python-docx (when installing), but the import name is docx (no python- prefix)
from docx import Document
from docx.shared import Inches, Pt  # Inches/Pt: helper classes for specifying sizes

# Document() creates a new blank Word document
doc = Document()

# -- Title and time --
# level=1 maps to Word's "Heading 1" (the largest heading)
doc.add_heading("Pine and Cypress Nursing Home Legionnaires' SitRep", level=1)
doc.add_paragraph(f"Report time: {report_time}")
doc.add_paragraph(f"Data source: legionella_outbreak.csv ({total} resident records)")

# -- Summary metrics table --
doc.add_heading("Summary metrics", level=2)
# add_table(rows=6, cols=2): create a 6-row, 2-column table
# style="Light Grid Accent 1": apply Word's built-in table style (light grid)
table = doc.add_table(rows=6, cols=2, style="Light Grid Accent 1")
metrics = [
    ("Total residents", str(total)),
    ("Infected", f"{infected} (attack rate {infected/total:.1%})"),
    ("Confirmed", str(confirmed)),
    ("Probable", str(probable)),
    ("Hospitalized", f"{hospitalized} (hospitalization rate {hospitalized/infected:.1%})"),
    ("Deaths", f"{deaths} (CFR {deaths/infected:.1%})"),
]
# enumerate() gives both the index i and the value (label, value)
for i, (label, value) in enumerate(metrics):
    # table.rows[i] gets row i, .cells[0] gets the first column
    table.rows[i].cells[0].text = label
    table.rows[i].cells[1].text = value

# -- Embed the epidemic curve --
doc.add_heading("Epidemic curve", level=2)
epicurve_buf.seek(0)  # reset the BytesIO read cursor (always seek(0) before each read)
# width=Inches(6): set the image width to 6 inches
doc.add_picture(epicurve_buf, width=Inches(6))

# -- Per-wing stats (header + data rows) --
doc.add_heading("Outbreak summary by wing", level=2)
# rows=len(wing_stats) + 1: number of data rows + 1 header row
wing_table = doc.add_table(
    rows=len(wing_stats) + 1, cols=5, style="Light Grid Accent 1"
)
headers = ["Wing", "Residents", "Infected", "AR%", "CFR%"]
for j, h in enumerate(headers):
    wing_table.rows[0].cells[j].text = h  # row 0 = header
for i, row in wing_stats.iterrows():
    wing_table.rows[i + 1].cells[0].text = str(row["label"])
    wing_table.rows[i + 1].cells[1].text = str(row["residents"])
    wing_table.rows[i + 1].cells[2].text = str(row["infected"])
    wing_table.rows[i + 1].cells[3].text = str(row["AR%"])
    wing_table.rows[i + 1].cells[4].text = str(row["CFR%"])

doc.save("output/sitrep_report.docx")
print("✅ Word report saved: output/sitrep_report.docx")


In [ ]:
# --- 8c: Presentation slides (PPTX) ---
from pptx import Presentation
from pptx.util import Inches, Pt  # Inches/Pt: helper classes for specifying position and size

prs = Presentation()  # create a new blank PPTX (defaults to 16:9 slides)

# -- Slide 1: title page --
# slide_layouts[0] is the "Title Slide" layout (title + subtitle placeholders)
slide1 = prs.slides.add_slide(prs.slide_layouts[0])
slide1.shapes.title.text = "Pine and Cypress Nursing Home Legionnaires' SitRep"
# placeholders[1] is the subtitle (index 0 = main title, index 1 = subtitle)
slide1.placeholders[1].text = f"Report time: {report_time}"

# -- Slide 2: key figures --
# slide_layouts[5] is the "Blank" layout with no placeholders
slide2 = prs.slides.add_slide(prs.slide_layouts[5])
# add_textbox(left, top, width, height): specify position and size with Inches
txBox = slide2.shapes.add_textbox(
    Inches(1), Inches(0.5), Inches(8), Inches(1),
)
txBox.text_frame.text = "Key summary metrics"
txBox.text_frame.paragraphs[0].font.size = Pt(28)
txBox.text_frame.paragraphs[0].font.bold = True

# Body text box (placed below the heading)
body = slide2.shapes.add_textbox(
    Inches(1), Inches(1.8), Inches(8), Inches(4),
)
tf = body.text_frame
tf.word_wrap = True  # allow word wrapping
kpi_lines = [
    f"Infected: {infected} (attack rate {infected/total:.1%})",
    f"Confirmed: {confirmed}   Probable: {probable}",
    f"Hospitalized: {hospitalized}   ICU: {icu}",
    f"Deaths: {deaths} (CFR {deaths/infected:.1%})",
]
for line in kpi_lines:
    p = tf.add_paragraph()   # add a new paragraph per line
    p.text = line
    p.font.size = Pt(20)
    p.space_after = Pt(12)   # 12 pt space after the paragraph

# -- Slide 3: epidemic curve --
slide3 = prs.slides.add_slide(prs.slide_layouts[5])
txBox3 = slide3.shapes.add_textbox(
    Inches(1), Inches(0.3), Inches(8), Inches(0.8),
)
txBox3.text_frame.text = "Epidemic curve (by onset date)"
txBox3.text_frame.paragraphs[0].font.size = Pt(24)
txBox3.text_frame.paragraphs[0].font.bold = True

epicurve_buf.seek(0)  # reset the BytesIO cursor to read the image data again
# add_picture(image, left, top, width, height): insert the image at a set position
slide3.shapes.add_picture(epicurve_buf, Inches(0.5), Inches(1.3), Inches(9), Inches(5))

# -- Slide 4: attack rate by wing --
slide4 = prs.slides.add_slide(prs.slide_layouts[5])
txBox4 = slide4.shapes.add_textbox(
    Inches(1), Inches(0.3), Inches(8), Inches(0.8),
)
txBox4.text_frame.text = "Outbreak summary by wing"
txBox4.text_frame.paragraphs[0].font.size = Pt(24)
txBox4.text_frame.paragraphs[0].font.bold = True

# add_table(...).table gets the table object (the chain returns a GraphicFrame; .table is the Table)
rows_n = len(wing_stats) + 1  # data rows + 1 header row
tbl = slide4.shapes.add_table(rows_n, 5, Inches(0.5), Inches(1.3), Inches(9), Inches(4)).table
for j, h in enumerate(["Wing", "Residents", "Infected", "AR%", "CFR%"]):
    tbl.cell(0, j).text = h  # row 0 = header
for i, row in wing_stats.iterrows():
    tbl.cell(i + 1, 0).text = str(row["label"])
    tbl.cell(i + 1, 1).text = str(row["residents"])
    tbl.cell(i + 1, 2).text = str(row["infected"])
    tbl.cell(i + 1, 3).text = str(row["AR%"])
    tbl.cell(i + 1, 4).text = str(row["CFR%"])

prs.save("output/sitrep_slides.pptx")
print("✅ Slides saved: output/sitrep_slides.pptx")


In [ ]:
# --- 8d: Formal PDF report (fpdf2) ---
# By default fpdf2 only has Latin fonts; showing CJK requires manually embedding a TTF/TTC font
import pathlib as _pl
from fpdf import FPDF

# -- CJK font detection --
# Scan the system font directories for a font whose name contains "CJK", "WenQuanYi", or "wqy"
cjk_font_path = None
for font_dir in ["/usr/share/fonts", "/usr/local/share/fonts"]:
    for fp in sorted(_pl.Path(font_dir).rglob("*")):
        if fp.suffix.lower() in {".ttf", ".ttc"} and (
            "CJK" in fp.name or "WenQuanYi" in fp.name or "wqy" in fp.name
        ):
            cjk_font_path = str(fp)
            break
    if cjk_font_path:
        break

pdf = FPDF()        # create a new PDF (defaults to A4 portrait)
pdf.add_page()      # you must add_page() first before you can write any content

# -- Font setup --
if cjk_font_path:
    # add_font("alias", "style", "font file path")
    # fpdf2 v2.5.1+ doesn't need uni=True (Unicode is supported automatically)
    pdf.add_font("CJK", "", cjk_font_path)
    pdf.set_font("CJK", size=16)
else:
    pdf.set_font("Helvetica", size=16)
    print("⚠️ No CJK font found; Chinese may not display. Please install fonts-noto-cjk")

# -- Title --
# cell(width, height, text, ...) is fpdf2's most basic content unit
# width=0 means "extend to the right margin" (auto-fill the page width)
# new_x="LMARGIN": next cell starts at the left margin; new_y="NEXT": moves to the next line
# align="C": center the text within the cell
pdf.cell(0, 12, text="Pine and Cypress Nursing Home Legionnaires' SitRep", new_x="LMARGIN", new_y="NEXT", align="C")
pdf.set_font_size(10)
pdf.cell(0, 8, text=f"Report time: {report_time}", new_x="LMARGIN", new_y="NEXT", align="C")
pdf.ln(8)  # ln(n): insert n points of blank line (layout spacing)

# -- Summary metrics --
pdf.set_font_size(13)
pdf.cell(0, 10, text="Summary metrics", new_x="LMARGIN", new_y="NEXT")
pdf.set_font_size(10)
kpi_lines = [
    f"Total residents: {total}",
    f"Infected: {infected} (attack rate {infected/total:.1%})",
    f"Confirmed: {confirmed}   Probable: {probable}",
    f"Hospitalized: {hospitalized} (hospitalization rate {hospitalized/infected:.1%})",
    f"Deaths: {deaths} (CFR {deaths/infected:.1%})",
]
for line in kpi_lines:
    pdf.cell(0, 7, text=line, new_x="LMARGIN", new_y="NEXT")
pdf.ln(5)

# -- Embed the epidemic curve --
# fpdf2's pdf.image() only accepts a "file path" string, not a BytesIO object
# Workaround: write the BytesIO contents to a temporary PNG, embed it, then delete it
pdf.set_font_size(13)
pdf.cell(0, 10, text="Epidemic curve", new_x="LMARGIN", new_y="NEXT")
epicurve_buf.seek(0)
epicurve_tmp = _pl.Path("output/epicurve_tmp.png")
epicurve_tmp.write_bytes(epicurve_buf.read())  # write the BytesIO data to disk
# pdf.w is the page width (~210 mm); subtracting 30 leaves 15 mm margins on each side
pdf.image(str(epicurve_tmp), w=pdf.w - 30)
epicurve_tmp.unlink()  # delete the temp file after embedding (clean up)
pdf.ln(5)

# -- Per-wing stats table (manually drawn gridded table) --
pdf.add_page()  # add a second page for the table
pdf.set_font_size(13)
pdf.cell(0, 10, text="Outbreak summary by wing", new_x="LMARGIN", new_y="NEXT")
pdf.set_font_size(9)

# col_widths defines each column's width (mm); the total should be less than the effective page width (~190 mm)
col_widths = [25, 25, 25, 30, 30]
headers = ["Wing", "Residents", "Infected", "AR%", "CFR%"]
# header row: border=1 draws all four borders
for w, h in zip(col_widths, headers):
    pdf.cell(w, 8, text=h, border=1, align="C")
pdf.ln()  # line break after the header

# data rows: _ means we don't need the index (just the value, row)
for _, row in wing_stats.iterrows():
    vals = [str(row["label"]), str(row["residents"]), str(row["infected"]),
            str(row["AR%"]), str(row["CFR%"])]
    for w, v in zip(col_widths, vals):
        pdf.cell(w, 7, text=v, border=1, align="C")
    pdf.ln()  # line break after each data row

pdf.output("output/sitrep_report.pdf")
print("✅ PDF report saved: output/sitrep_report.pdf")


## Step 9: Schedule automatic updates

Schedule the SitRep pipeline to run automatically — have the computer run the script every morning at 9 a.m., so your supervisor sees the latest daily report the moment they open their inbox.

| Platform | Tool | How to configure |
|------|------|--------|
| macOS | launchd (recommended) | plist XML + `launchctl load` |
| Linux | cron | `crontab -e` |
| Windows | Task Scheduler / schtasks | GUI or command line |

> 📖 For the full scheduling tutorial (with config file examples for each platform), see the text version of Ch04 Step 9.
> Here we only show how to first test in the notebook whether the script runs correctly.


In [ ]:
# --- Step 9: Test manually before scheduling (confirm the script runs) ---
# Before setting up the schedule, run the script once to confirm it works
import subprocess, sys, os

# Find the project root (three levels up from the notebook: notebooks/ → chapters/ → book/ → root)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
script_path = os.path.join(project_root, "notebooks", "run_sitrep.py")

print(f"Project root: {project_root}")
print(f"Script path: {script_path}")
print(f"Script exists: {os.path.exists(script_path)}")
print()

result = subprocess.run(
    [sys.executable, script_path],  # sys.executable = the absolute path of the current Python
    capture_output=True,  # capture stdout/stderr (don't print straight to the screen)
    text=True,            # return the output as a string (not bytes)
    cwd=project_root,     # run in the project root (simulating the scheduler environment)
)

print("=== Script output ===")
print(result.stdout)
if result.returncode != 0:
    print("=== Error message ===")
    print(result.stderr)
else:
    print("✅ The script ran successfully — you can safely set up the schedule!")


## Summary

You've completed the automated production pipeline for a standard SitRep:

| Step | Content | Skills learned |
|------|------|------------|
| 1 | Read in and prepare | Date conversion, derived columns |
| 2 | Summary metrics | Attack rate, CFR, hospitalization rate |
| 3 | Person | Age distribution, comorbidity prevalence |
| 4 | Time | Epidemic curve, peak day |
| 5 | Place | Attack-rate-by-wing table |
| 6 | Case classification | Confirmed / probable / not-a-case strata |
| 7 | Functionization | A rerunnable `generate_sitrep()` |
| 8 | Professional report | Plotly dashboard, DOCX, PPTX, PDF |
| 9 | Scheduled auto-update | launchd / cron / Task Scheduler |

In the next chapter (Ch05), we return to the question Ch03 left open — is the high RR for shower use a genuine causal effect, or was it inflated by confounders?